In [26]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUTS = Path('../outputs/ongoing/swift_apex_benchmark')

rows = []
for result_file in sorted(OUTPUTS.rglob('result_*.json')):
    parts = result_file.parts
    method = parts[-4]
    task   = parts[-3]
    seed   = parts[-2].replace('seed_', '')
    with open(result_file, encoding='utf-8') as f:
        d = json.load(f)
    rows.append({
        'method':       method,
        'task':         task.replace('bbh_', ''),
        'seed':         int(seed),
        'dev_score':    d.get('best_score', 0.0),
        'test_score':   d.get('test_score', 0.0),
        'total_time':   d.get('total_time', 0.0),
        'llm_calls':    d.get('llm_usage', {}).get('total_calls', 0),
        'total_tokens': d.get('llm_usage', {}).get('total_tokens', 0),
        'num_iters':    d.get('num_iterations', 0),
    })

df = pd.DataFrame(rows)
print(f'{len(df)} runs loaded — methods: {sorted(df.method.unique())} | tasks: {sorted(df.task.unique())}')
df.head()

64 runs loaded — methods: ['apex', 'capo', 'gaapo', 'see', 'swift'] | tasks: ['causal_judgement', 'disambiguation_qa', 'formal_fallacies', 'hyperbaton', 'logical_deduction_five_objects']


,method,task,seed,dev_score,test_score,total_time,llm_calls,total_tokens,num_iters
0,apex,causal_judgement,123,0.733333,0.566667,200.97,604,288942,5
1,apex,causal_judgement,42,0.800000,0.516667,236.47,604,336975,5
2,apex,causal_judgement,7,0.766667,0.616667,206.86,604,320312,5
3,apex,disambiguation_qa,123,0.533333,0.416667,162.45,564,185953,5
4,apex,disambiguation_qa,42,0.566667,0.466667,212.81,794,240327,5


## Test Score — mean ± std across seeds

In [27]:
agg = df.groupby(['method', 'task'])['test_score'].agg(['mean', 'std', 'count']).reset_index()
agg['std'] = agg['std'].fillna(0.0)
agg['score'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)

pivot = agg.pivot(index='task', columns='method', values='score')

# Also compute a mean column per method (macro-average over tasks)
mean_per_method = df.groupby('method')['test_score'].mean()
macro_row = {m: f"{mean_per_method[m]:.3f}" for m in pivot.columns}
pivot.loc['**macro avg**'] = macro_row

pd.set_option('display.max_colwidth', 20)
print('Test score (mean ± std, 3 seeds)')
pivot

Test score (mean ± std, 3 seeds)


method,apex,capo,gaapo,see,swift
task,,,,,
causal_judgement,0.567 ± 0.050,0.544 ± 0.079,0.600 ± 0.073,0.583 ± 0.060,0.561 ± 0.098
disambiguation_qa,0.294 ± 0.256,0.439 ± 0.063,0.000 ± 0.000,0.450 ± 0.029,0.244 ± 0.171
formal_fallacies,0.000 ± 0.000,0.383 ± 0.355,0.000 ± 0.000,0.122 ± 0.212,0.000 ± 0.000
hyperbaton,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.194 ± 0.337
logical_deduction_five_objects,0.333 ± 0.000,NaN,NaN,NaN,0.394 ± 0.226
**macro avg**,0.224,0.342,0.150,0.289,0.279


## Dev vs Test Score (overfitting check)

In [28]:
gap = df.groupby(['method', 'task']).agg(
    dev_mean=('dev_score', 'mean'),
    test_mean=('test_score', 'mean'),
).reset_index()
gap['gap'] = gap['dev_mean'] - gap['test_mean']

gap_pivot = gap.pivot(index='task', columns='method', values='gap').round(3)
print('Dev - Test gap (positive = overfit to dev)')
gap_pivot.style.background_gradient(cmap='RdYlGn_r', axis=None)

Dev - Test gap (positive = overfit to dev)


method,apex,capo,gaapo,see,swift
task,,,,,
causal_judgement,0.200000,0.200000,0.133000,0.172000,0.217000
disambiguation_qa,0.072000,0.050000,0.000000,0.006000,0.089000
formal_fallacies,0.000000,-0.028000,0.000000,-0.044000,0.000000
hyperbaton,0.000000,0.000000,0.000000,0.000000,0.083000
logical_deduction_five_objects,-0.033000,nan,nan,nan,-0.028000


## Efficiency — LLM calls and wall-clock time per run

In [29]:
eff = df.groupby('method').agg(
    avg_calls=('llm_calls', 'mean'),
    avg_tokens=('total_tokens', 'mean'),
    avg_time_s=('total_time', 'mean'),
    avg_iters=('num_iters', 'mean'),
    avg_test_score=('test_score', 'mean'),
).round(1)
eff['score_per_100_calls'] = (eff['avg_test_score'] / (eff['avg_calls'] / 100)).round(3)
eff

,avg_calls,avg_tokens,avg_time_s,avg_iters,avg_test_score,score_per_100_calls
method,,,,,,
apex,716.5,266840.1,226.9,4.5,0.2,0.028
capo,1860.5,766624.9,757.7,4.8,0.3,0.016
gaapo,530.4,270520.8,231.1,4.2,0.2,0.038
see,671.6,459463.7,374.0,3.0,0.3,0.045
swift,565.4,373575.5,254.8,3.0,0.3,0.053


## Per-task best method

In [30]:
best = agg[agg['task'] != '**macro avg**'].copy()
best['mean_val'] = best['mean']
idx = best.groupby('task')['mean_val'].idxmax()
best_per_task = best.loc[idx, ['task', 'method', 'mean', 'std']].rename(
    columns={'mean': 'test_mean', 'std': 'test_std', 'method': 'best_method'}
).set_index('task')
best_per_task

,best_method,test_mean,test_std
task,,,
causal_judgement,gaapo,0.600000,0.072648
disambiguation_qa,see,0.450000,0.028868
formal_fallacies,capo,0.383333,0.354730
hyperbaton,swift,0.194444,0.336788
logical_deduction_five_objects,swift,0.394444,0.226282


## Raw data

In [31]:
df.sort_values(['task', 'method', 'seed']).reset_index(drop=True)

,method,task,seed,dev_score,test_score,total_time,llm_calls,total_tokens,num_iters
0,apex,causal_judgement,7,0.766667,0.616667,206.86,604,320312,5
1,apex,causal_judgement,42,0.800000,0.516667,236.47,604,336975,5
2,apex,causal_judgement,123,0.733333,0.566667,200.97,604,288942,5
3,capo,causal_judgement,7,0.733333,0.516667,652.14,1577,878373,4
4,capo,causal_judgement,42,0.766667,0.483333,651.56,1572,852902,4
...,...,...,...,...,...,...,...,...,...
59,swift,hyperbaton,123,0.833333,0.583333,236.11,509,300626,3
60,apex,logical_deductio...,42,0.300000,0.333333,259.92,884,319033,5
61,swift,logical_deductio...,7,0.533333,0.516667,226.59,509,365898,3
62,swift,logical_deductio...,42,0.200000,0.133333,252.39,591,374303,3
